In [2]:
import pandas as pd

def _to_str_clean(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    return s

def _normalize_whitespace(s: str) -> str:
    return " ".join(str(s).strip().split())

def distinct_join(values, sep=", "):
    seen = set()
    out = []
    for v in values:
        s = _to_str_clean(v)
        if s is None:
            continue
        key = _normalize_whitespace(s)
        if key not in seen:
            seen.add(key)
            out.append(key)
    return sep.join(out)

def dates_join(values, sep=", ", sort_dates=True):
    raw = [_to_str_clean(v) for v in values]
    raw = [r for r in raw if r is not None]
    if not raw:
        return ""

    # ✅ pandas-compatible mixed parsing (newer pandas supports format="mixed")
    try:
        dt = pd.to_datetime(raw, errors="coerce", format="mixed")
    except TypeError:
        # fallback for older pandas that doesn't support format="mixed"
        dt = pd.to_datetime(raw, errors="coerce")

    parsed = []
    unparsed = []
    for orig, d in zip(raw, dt):
        if pd.isna(d):
            unparsed.append(orig)
        else:
            parsed.append(d)

    if sort_dates and parsed:
        parsed = sorted(parsed)

    formatted = [f"{d.month}/{d.day}/{d.year}" for d in parsed]
    return distinct_join(formatted + unparsed, sep=sep)

def long_to_wide(
    df: pd.DataFrame,
    address_col="Address",
    subcategory_col="Subcategory",
    date_col="Date",
    units_col="Total Units",
    price_col="Sales Price",
    sort_dates=True
):
    for col in [address_col, subcategory_col, date_col]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df = df.copy()

    df[address_col] = df[address_col].map(_to_str_clean).map(lambda x: _normalize_whitespace(x) if x else None)
    df[subcategory_col] = df[subcategory_col].map(_to_str_clean).map(lambda x: _normalize_whitespace(x) if x else None)
    df[date_col] = df[date_col].map(_to_str_clean)

    g_dates = (
        df.groupby([address_col, subcategory_col], dropna=False)[date_col]
          .apply(lambda s: dates_join(s.tolist(), sort_dates=sort_dates))
          .reset_index(name="__dates__")
    )

    wide_events = (
        g_dates.pivot(index=address_col, columns=subcategory_col, values="__dates__")
               .reset_index()
    )

    if units_col in df.columns:
        units_agg = (
            df.groupby(address_col, dropna=False)[units_col]
              .apply(lambda s: distinct_join(s.tolist()))
              .reset_index(name=units_col)
        )
        wide_events = wide_events.merge(units_agg, on=address_col, how="left")

    if price_col in df.columns:
        price_agg = (
            df.groupby(address_col, dropna=False)[price_col]
              .apply(lambda s: distinct_join(s.tolist()))
              .reset_index(name=price_col)
        )
        wide_events = wide_events.merge(price_agg, on=address_col, how="left")

    cols = list(wide_events.columns)
    subcat_cols = [c for c in cols if c not in {address_col, units_col, price_col}]
    subcat_cols = sorted(subcat_cols)

    ordered = [address_col]
    if price_col in cols:
        ordered.append(price_col)
    if units_col in cols:
        ordered.append(units_col)
    ordered += subcat_cols

    return wide_events[ordered]

# RUN
input_path = "input.csv"
df = pd.read_csv(input_path)

wide = long_to_wide(df)
wide.to_csv("output_wide.csv", index=False)
print("Wrote: output_wide.csv")

Wrote: output_wide.csv


In [4]:
import pandas as pd
import re

input_csv = "output_wide.csv"
output_xlsx = "output_wide_split.xlsx"

df = pd.read_csv(input_csv, dtype=str, low_memory=False)

# ----------------------------
# Basic helpers
# ----------------------------
def _clean_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()

def _is_nonempty(x) -> bool:
    return _clean_cell(x) != ""

# ----------------------------
# Splitters
# ----------------------------

# Sales Price: extract money tokens WITHOUT splitting thousands commas
_price_token_re = re.compile(r"\$?\s*-?\d[\d,]*(?:\.\d+)?")

def split_sales_price_cell(s: str):
    s = _clean_cell(s)
    if not s:
        return []
    toks = [t.strip() for t in _price_token_re.findall(s) if t.strip()]
    return toks if toks else [s]

_units_token_re = re.compile(r"-?\d+(?:\.\d+)?")

def split_total_units_cell(s: str):
    s = _clean_cell(s)
    if not s:
        return []
    toks = _units_token_re.findall(s)
    return toks if toks else [s]

def split_generic_cell(s: str):
    s = _clean_cell(s)
    if not s:
        return []
    return [p.strip() for p in s.split(",") if p.strip()]

# ----------------------------
# Expand any column if any row has >1 values
# ----------------------------
def expand_column_any(df, col, splitter):
    if col not in df.columns:
        return df

    lists = df[col].apply(splitter)
    max_len = int(lists.map(len).max() or 0)

    if max_len <= 1:
        return df

    expanded = pd.DataFrame(
        lists.tolist(),
        index=df.index,
        columns=[f"{col} {i}" for i in range(1, max_len + 1)]
    )

    insert_at = df.columns.get_loc(col) + 1
    left = df.iloc[:, :insert_at]
    right = df.iloc[:, insert_at:]
    df2 = pd.concat([left, expanded, right], axis=1)

    return df2.drop(columns=[col])

# ----------------------------
# Drop fully empty columns in a sheet
# ----------------------------
def drop_fully_empty_columns(sheet_df):
    sheet_df = sheet_df.copy().replace("", pd.NA)
    return sheet_df.dropna(axis=1, how="all")

def ensure_min_one_col(sheet_df, base_name: str):
    cols = [c for c in sheet_df.columns if c == base_name or c.startswith(base_name + " ")]
    if not cols:
        sheet_df[f"{base_name} 1"] = pd.NA
    return sheet_df

# ----------------------------
# 1) Expand multi-values in ALL columns
# ----------------------------
df = expand_column_any(df, "Total Units", split_total_units_cell)
df = expand_column_any(df, "Sales Price", split_sales_price_cell)

for col in list(df.columns):
    if col == "Address":
        continue
    if col.startswith("Total Units") or col.startswith("Sales Price"):
        continue
    df = expand_column_any(df, col, split_generic_cell)

# ----------------------------
# 2) Classification based on COLUMN NAMES + whether those cols have values
# ----------------------------

# Robust patterns (NO \b around + / -)
pat_sfd = re.compile(r"SFD|Single\s*Family", re.IGNORECASE)
pat_24  = re.compile(r"2\s*-\s*4|2\s*to\s*4|2\s*–\s*4", re.IGNORECASE)
pat_5p  = re.compile(r"5\s*\+|5\+|5\s*plus|5\s*or\s*more", re.IGNORECASE)

# Identify event-like columns (everything except Address, Total Units*, Sales Price*)
non_event_like = {"Address"}
non_event_like |= {c for c in df.columns if c.startswith("Total Units")}
non_event_like |= {c for c in df.columns if c.startswith("Sales Price")}
event_cols = [c for c in df.columns if c not in non_event_like]

sfd_cols = [c for c in event_cols if pat_sfd.search(c)]
u24_cols = [c for c in event_cols if pat_24.search(c)]
u5p_cols = [c for c in event_cols if pat_5p.search(c)]

def has_any_value(row, cols):
    if not cols:
        return False
    for c in cols:
        if _is_nonempty(row.get(c, "")):
            return True
    return False

def classify_row(row):
    # priority: 5+ > 2-4 > SFD > Other
    if has_any_value(row, u5p_cols):
        return "5+ Units"
    if has_any_value(row, u24_cols):
        return "2-4 Units"
    if has_any_value(row, sfd_cols):
        return "SFD"
    return "Other"

# ----------------------------
# DEBUG: confirm columns + counts
# ----------------------------
print("Detected columns:")
print("  5+ cols:", len(u5p_cols))
print("  2-4 cols:", len(u24_cols))
print("  SFD cols:", len(sfd_cols))

# If 5+ cols is 0, print a few event col names to see what they look like
if len(u5p_cols) == 0:
    print("\nNo 5+ columns matched. Here are example event columns:")
    for c in event_cols[:25]:
        print("   -", c)

df["__Group__"] = df.apply(classify_row, axis=1)
print("\nRow counts by group:")
print(df["__Group__"].value_counts(dropna=False))

# ----------------------------
# 3) Split into sheets
# ----------------------------
df_all   = df.drop(columns="__Group__")
df_5p    = df[df["__Group__"] == "5+ Units"].drop(columns="__Group__")
df_24    = df[df["__Group__"] == "2-4 Units"].drop(columns="__Group__")
df_sfd   = df[df["__Group__"] == "SFD"].drop(columns="__Group__")
df_other = df[df["__Group__"] == "Other"].drop(columns="__Group__")

# ----------------------------
# 4) Finalize per-sheet
# ----------------------------
def finalize_sheet(sheet_df):
    sheet_df = drop_fully_empty_columns(sheet_df)

    # must exist even if empty
    sheet_df = ensure_min_one_col(sheet_df, "Sales Price")
    sheet_df = ensure_min_one_col(sheet_df, "Total Units")
    return sheet_df

df_all   = finalize_sheet(df_all)
df_sfd   = finalize_sheet(df_sfd)
df_24    = finalize_sheet(df_24)
df_5p    = finalize_sheet(df_5p)
df_other = finalize_sheet(df_other)

# ----------------------------
# 5) Write
# ----------------------------
with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    df_all.to_excel(writer, sheet_name="All", index=False)
    df_sfd.to_excel(writer, sheet_name="SFD", index=False)
    df_24.to_excel(writer, sheet_name="2-4 Units", index=False)
    df_5p.to_excel(writer, sheet_name="5+ Units", index=False)
    df_other.to_excel(writer, sheet_name="Other", index=False)

print("\nWrote:", output_xlsx)

Detected columns:
  5+ cols: 22
  2-4 cols: 19
  SFD cols: 47

Row counts by group:
__Group__
SFD          7797
Other        2754
2-4 Units    1821
5+ Units      864
Name: count, dtype: int64

Wrote: output_wide_split.xlsx
